# Guided Lab：建立具有短期記憶的客服 Agent

**時間：20 分鐘。** 延續 [短期記憶講義](memory_1.md)，建立電商客服 Agent，查詢 A1024 後接續詢問「什麼時候會到？」。

本 lab 同時練習程式撰寫程序與「概念機器」：在心中追蹤系統如何載入對話狀態、加入新訊息、執行任務及保存狀態。

## 學習目標

1. 建立記憶儲存器，並交給 Agent 使用。
2. 使用對話緒設定進行兩輪呼叫，每次只傳入新增的使用者訊息。
3. 透過訊息歷史解釋第二輪為何能取得訂單資訊。
4. 預測全新對話緒可取得哪些資訊。


## 活動安排

| 時間 | 活動 |
|---|---|
| 0–2 分鐘 | 閱讀情境與已提供的元件 |
| 2–10 分鐘 | 完成六個填空，執行第一輪 |
| 10–15 分鐘 | 預測第二輪資料流，執行並核對訊息歷史 |
| 15–18 分鐘 | 預測並比較全新對話緒 |
| 18–20 分鐘 | 完成解釋題並繳交 |


## 使用方式

由上往下執行；將六個 `__________` 替換為適當程式碼。填空完成前，這些儲存格無法正確執行。

先填寫預測，再執行對應程式。

若需要重做整個實驗，從設定及建立 Agent 的儲存格重新執行，再依序向下操作；這會建立新的記憶儲存器。只重跑呼叫儲存格會繼續累積原對話緒的訊息。


## 已提供的元件

- 變數 `model_name` 指定模型；
  - 可替換成自己的模型名稱
  - Format: "provider:model_name" (e.g., "openai:gpt-4o-mini" or "google:gemini-1.5-pro").
- 函數 `get_order_status` 從固定字典取得訂單資料；
- 變數 `tools` 是 Agent 可使用的工具清單；
- 變數 `system_prompt` 指定客服的回應方式。

工具延續前章的 A1024、B2048 與 C4096 資料，日期為固定的歷史測試值。模型根據工具回傳的狀態及日期回答。


In [ ]:
# Load API Key 
from getpass import getpass
import os
from pathlib import Path

from dotenv import load_dotenv
dotenv_path = Path('your_env_file_path/.env')  # Replace with the path to your .env file
# load_dotenv(dotenv_path=dotenv_path)  # take environment variables from .env.
load_dotenv()  # take environment variables from .env.

# 可於執行時隱藏輸入。
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("請輸入 OpenAI API key：")

In [ ]:

from langchain.tools import tool

# 教師預先設定模型；學生直接使用。
model_name = "openai:gpt-5-nano"

# 可使用課堂環境預先設定的金鑰，或在執行時隱藏輸入。
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("請輸入 OpenAI API key：")

@tool
def get_order_status(order_id: str) -> dict:
    """依訂單編號查詢模擬的配送狀態與預計送達日期。"""
    orders = {
        "A1024": {"status": "Shipped", "estimated_delivery": "2023-07-25"},
        "B2048": {"status": "Processing", "estimated_delivery": "2023-07-30"},
        "C4096": {"status": "Delivered", "estimated_delivery": "2023-07-20"},
    }
    return orders.get(
        order_id,
        {"status": "Order ID not found", "estimated_delivery": None},
    )

tools = [get_order_status]

system_prompt = """
你是電商客服助理，請以繁體中文簡潔回答訂單問題。
從目前使用者訊息或同一對話緒的訊息歷史取得訂單編號。
若無法確定訂單編號，先詢問使用者，不要猜測或呼叫工具。
使用 get_order_status 查詢訂單目前的配送狀態，原樣傳入訂單編號。
根據工具結果回答；後續問題可使用同一對話緒中已有的工具結果。
若訂單編號不存在，請使用者確認編號。
訂單資料為固定的教學測試資料，請照資料回答日期。
與訂單無關的問題不需要呼叫工具。
"""

print("課堂元件已備妥，可以開始建立 Agent。")


## 任務一：建立具有短期記憶的 Agent（①–③）

`InMemorySaver` 是記憶儲存器，負責將檢查點保存在目前 Python 程序的記憶體中。檢查點是某個執行時刻的狀態快照。建立 Agent 時，使用 `checkpointer` 參數傳入儲存器。

請完成①–③。執行後會得到 `checkpointer` 與 `agent`，尚未開始對話。


In [ ]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

# ① 建立記憶儲存器
checkpointer = __________

# ② 建立 Agent
agent = __________(
    model=model_name,
    tools=tools,
    system_prompt=system_prompt,
    # ③ 傳入記憶儲存器
    checkpointer=__________
)


## 任務二：指定對話緒並執行第一輪（④–⑤）

`thread_id` 是對話緒的識別碼。呼叫 Agent 時，將它放在設定字典的 `configurable` 欄位內，再透過 `config` 傳入。

請完成④–⑤。這一輪輸入訂單編號 A1024，程式會印出 Agent 最後的回覆。


In [ ]:
thread_config = {
    "configurable": {
        # ④ 填入指定對話緒的欄位名稱
        "__________": "order-a1024"
    }
}

response_1 = agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "請查詢訂單 A1024 的配送狀態。"}
        ]
    },
    # ⑤ 傳入對話緒設定
    config=__________
)

print(response_1["messages"][-1].content)


## 執行第二輪前：先追蹤資料流

第二輪只傳入「什麼時候會到？」。請先預測 Agent 從哪裡取得 A1024，並在下面的 Markdown 儲存格記錄。

追蹤順序：**指定對話緒 → 載入已保存的狀態 → 加入新訊息 → Agent 執行 → 保存更新後的狀態。**


### 第二輪的預測與追蹤（請編輯本儲存格）

| 步驟 | 我的預測 |
|---|---|
| 指定對話緒：使用哪個識別碼？ | 待填 |
| 載入狀態：能取回哪些舊訊息？ | 待填 |
| 加入新訊息：本輪新增什麼？ | 待填 |
| Agent 執行：如何理解後續問題？ | 待填 |
| 保存狀態：歷史將增加什麼？ | 待填 |


## 任務三：執行第二輪（⑥）

請完成⑥，使用與第一輪相同的對話緒設定。輸入只包含本輪新增的使用者訊息，先前的歷史由系統透過 checkpointer 載入。


In [ ]:
response_2 = agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "什麼時候會到？"}
        ]
    },
    # ⑥ 使用相同的對話緒設定
    config=__________
)

print(response_2["messages"][-1].content)


## 核對：觀察第二輪回傳的訊息歷史

以下是教師提供的顯示程式。`response_2["messages"]` 包含第二輪回傳狀態中的訊息歷史。

顯示的類型中，`human` 是使用者訊息，`ai` 是模型訊息，`tool` 是工具結果。AI 訊息若提出工具呼叫請求，程式會另外列出請求內容。

找出第一輪含 A1024 的使用者訊息、第一輪工具結果，以及第二輪的「什麼時候會到？」與回覆，核對你的預測。回傳歷史呈現保存結果；載入過程的說明需結合前面的資料流。


In [ ]:
# 教師提供：列出第二輪回傳狀態中的訊息。
for index, message in enumerate(response_2["messages"], start=1):
    print(f"{index}. [{message.type}] {message.content}")
    # 有些 AI 訊息以工具呼叫請求為主，文字內容可能為空。
    if getattr(message, "tool_calls", None):
        print("   工具呼叫請求：", message.tool_calls)


### 執行後核對（請編輯本儲存格）

- 第一輪含 A1024 的使用者訊息編號：待填。
- 第一輪工具結果的訊息編號：待填。
- 第二輪「什麼時候會到？」的訊息編號：待填。
- 預測與實際結果是否一致？如有不同，簡述差異：待填。


## 任務四：比較全新對話緒

下一個儲存格使用相同 Agent 與 checkpointer，只將 `thread_id` 改為尚未使用的 `new-conversation`，輸入仍是「什麼時候會到？」。

**先預測，再執行：**新對話緒是否包含 A1024 的歷史？Agent 可能如何回覆？


### 新對話緒的預測（請編輯本儲存格）

- 是否包含 A1024 的歷史？原因是什麼？待填。
- 預測 Agent 的處理方式：待填。


In [ ]:
# 教師提供：改用全新對話緒，Agent 與 checkpointer 保持相同。
new_thread_config = {
    "configurable": {
        "thread_id": "new-conversation"
    }
}

response_3 = agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "什麼時候會到？"}
        ]
    },
    config=new_thread_config
)

print("全新對話緒的回覆：")
print(response_3["messages"][-1].content)

print("\n全新對話緒的訊息歷史：")
for index, message in enumerate(response_3["messages"], start=1):
    print(f"{index}. [{message.type}] {message.content}")
    if getattr(message, "tool_calls", None):
        print("   工具呼叫請求：", message.tool_calls)


### 比較紀錄（請編輯本儲存格）

| 觀察項目 | `order-a1024` 的第二輪 | 全新的 `new-conversation` |
|---|---|---|
| 是否包含先前 A1024 的查詢訊息 | 待填 | 待填 |
| 回覆的處理方式 | 待填 | 待填 |

根據兩段訊息歷史，說明「載入狀態」這一步取得的資訊如何改變：待填。


## 最後解釋與繳交

第二次呼叫沒有傳入 A1024，Agent 為什麼仍能知道訂單編號？請用 `thread_id`、checkpointer 與訊息歷史說明。若改用全新的 `thread_id`，哪一步取得的資訊會改變？


### 我的解釋（請編輯本儲存格）

待填。

### 繳交前確認

- [ ] 已完成六個填空，並執行第一、第二輪呼叫。
- [ ] 已完成第二輪預測與訊息歷史核對。
- [ ] 已完成新對話緒的預測與比較紀錄。
- [ ] 已完成最後解釋，儲存並繳交本 notebook。


## 參考資料

- [課程講義：Agent 的短期記憶](memory_1.md)
- [本 lab 設計方案](guided_lab_design_for_short_term_memory.md)
- [LangChain 官方文件：Short-term memory](https://docs.langchain.com/oss/python/langchain/short-term-memory)

官方 API 文件查核日期：2026-10-04。
